In [1]:
import os
import sys
sys.path.append("/workspaces/dev/app")
os.chdir("/workspaces/dev")

In [2]:
import asyncio
import websockets
import msgpack
import numpy as np
import base64
import requests
import librosa
from IPython.display import Audio
from datetime import datetime
import json

In [3]:
from util.util import compress_to_opus, np_to_wav

In [4]:
SAMPLE_RATE = 16000
# BASE_URL = "wss://api.sangjeong.com:8080"
BASE_SOCKET_URL = "ws://localhost:8000"
BASE_URL = "http://localhost:8000"

In [5]:
original_audio, sr = librosa.load(".data/boda.wav", sr=SAMPLE_RATE)
# original_audio, sr = librosa.load(".data/news_with_english.wav", sr=SAMPLE_RATE)

In [6]:
total_samples = len(original_audio)

segments = []
pos = 0
while pos < total_samples:
    rand_len = int(np.random.normal(loc=4000, scale=400))
    rand_len = np.clip(rand_len, 3500, 4500)
    end = min(pos + rand_len, total_samples)

    chunk = original_audio[pos:end]
    segments.append(chunk)
    pos = end

In [7]:
idx = 0

In [ ]:
audio = segments[idx]
idx += 1
Audio(audio, rate=SAMPLE_RATE)

In [ ]:
def make_byte_json(param):
  text = json.dumps(param).encode("utf-8")
  length = len(text).to_bytes(4, byteorder="big")
  metadata = length + text
  return metadata

In [ ]:
def make_byte_audio(audio):
  if audio is None:
    return b''
  audio = np_to_wav(audio, SAMPLE_RATE)
  audio, _ = compress_to_opus(audio)
  return audio

In [ ]:
raise Exception("stop")

In [ ]:
async with websockets.connect(
    BASE_SOCKET_URL + "/asr/ws", ping_interval=30, ping_timeout=None
) as websocket:
    output = []

    async def send():
        for segment in segments:
            params = {
                "type": "asr",
                "userId": "0",
                "groupId": "0",
                "chunkId": 0,
                "timestamp": datetime.now().timestamp(),
            }
            param = make_byte_json(params)
            audio = make_byte_audio(segment)
            payload = param + audio
            await websocket.send(payload)

        params = {
            "type": "asr_done",
            "userId": "0",
            "groupId": "0",
            "chunkId": 0,
            "timestamp": datetime.now().timestamp(),
        }
        payload = make_byte_json(params)
        await websocket.send(payload)

    async def receive():
        while True:
            result = await websocket.recv()
            response = msgpack.unpackb(result, raw=False)
            print(response)
            if response and response["completed"]:
                output.append(response["completed"])

    await asyncio.gather(send(), receive())

In [ ]:
for ss in output:
    for s in ss:
        print(s["text"])
        print(f"\t start:{s['words'][0]['start']}, end:{s['words'][-1]['end']}")

In [ ]:
raise Exception("stop")

In [ ]:
refer_dict = {
    0: [(148061, 339589), (338760, 436145), (433746, 517335)],
    1: [(1013274, 1068548), (1071733, 1209101), (1209858, 1344749)],
    2: [(2563542, 2662059), (4373060, 4438192), (4570430, 4625790)],
    3: [(880324, 938541), (7042377, 7096457), (7106339, 7156845)],
    4: [(944284, 1009464), (1802918, 1921616), (1921626, 2053013)],
}

In [ ]:
embedding_refer_dict = {0: [], 1: [], 2: [], 3: [], 4: []}
for idx, ts in refer_dict.items():
    for s in ts:
        audio = original_audio[s[0] : s[1]]
        byts = np_to_wav(audio, SAMPLE_RATE)
        res = requests.post(
            BASE_URL + "/asr/embedding",
            data=byts,
            headers={"Content-Type": "application/octet-stream"},
        )
        embedding_refer_dict[idx].append(res.content)

In [ ]:
# print(np.frombuffer(embedding_refer_dict[0][0], dtype=np.float32).astype(np.float16))

In [ ]:
# import logging
# logging.basicConfig(level=logging.DEBUG)
# logger = logging.getLogger("websockets.protocol")
# logger.setLevel(logging.DEBUG)
USER_ID = "0"
GROUP_ID = "0"
async with websockets.connect(
    BASE_SOCKET_URL + "/asr/ws", ping_interval=30, ping_timeout=None
) as websocket:
    output = []

    async def send():
        param = {
            "type": "diarization_refer",
            "userId": USER_ID,
            "groupId": GROUP_ID,
            "chunkId": 0,
            "timestamp": datetime.now().timestamp(),
        }
        param = make_byte_json(param)
        refer_dict = {0: 3, 1: 3, 2: 3, 3: 3, 4: 3}
        payload = param + make_byte_json(refer_dict)
        for idx, length in refer_dict.items():
            for i in range(length):
                payload += embedding_refer_dict[idx][i]
        await websocket.send(payload)

        params = {
            "type": "metadata",
            "userId": USER_ID,
            "groupId": GROUP_ID,
            "chunkId": 0,
            "timestamp": datetime.now().timestamp(),
        }
        param = make_byte_json(params)
        metadata = {
            "agenda": ["자기소개", "각자 조사 자료 설명", "마무리"],
            "num_people": 5,
            "meeting_topic": "신기한 과학 소개 및 설명",
        }
        param += make_byte_json(metadata)
        await websocket.send(param)

        for idx, segment in enumerate(segments):
            await asyncio.sleep(0)
            params = {
                "type": "diarization_asr",
                "userId": USER_ID,
                "groupId": GROUP_ID,
                "chunkId": 0,
                "timestamp": datetime.now().timestamp(),
            }
            params = make_byte_json(params)
            audio = make_byte_audio(segment)
            payload = params + audio
            await websocket.send(payload)
            await asyncio.sleep(0.01)

            if idx != 0 and idx % 1000 == 0:
                params = {
                    "type": "context",
                    "userId": USER_ID,
                    "groupId": GROUP_ID,
                    "chunkId": 0,
                    "timestamp": datetime.now().timestamp(),
                }
                param = make_byte_json(params)
                await websocket.send(param)

        params = {
            "type": "context_done",
            "userId": USER_ID,
            "groupId": GROUP_ID,
            "chunkId": 0,
            "timestamp": datetime.now().timestamp(),
        }
        payload = make_byte_json(params)
        await websocket.send(payload)

        print("-------- send done --------")

    async def receive():
        while True:
            await asyncio.sleep(0)
            result = await websocket.recv()
            response = msgpack.unpackb(result, raw=False)
            print(response)
            if response and "completed" in response and response["completed"]:
                output.append(response["completed"])

    await asyncio.gather(send(), receive())

In [ ]:
for ss in output:
  for s in ss:
    print(s["text"])
    print(f"\t start:{s['words'][0]['start']}, end:{s['words'][-1]['end']}")